# Notebook 02

# Preparación e integración de diccionarios

**Objetivo:** Convertir los códigos numéricos de razas, colores y estados en nombres descriptivos para facilitar el análisis exploratorio y la construcción del dashboard en Power BI.

El conjunto de datos de PetFinder utiliza códigos numéricos para representar razas, colores y estados.

En esta etapa se incorporan los archivos de referencia (`breed_labels.csv`, `color_labels.csv` y `state_labels.csv`) para convertir dichos códigos en nombres descriptivos, conservando también los identificadores originales.

#### Importación de libreria

In [1]:
import pandas as pd

#### Carga del conjunto de datos preparado

Se importa el dataset generado en el Notebook 01, el cual contiene únicamente registros de gatos y las columnas renombradas al español.

In [2]:
df_limpio = pd.read_csv("dataset_gatos_limpio.csv")

print("Dimensiones:", df_limpio.shape)
display(df_limpio.head())

Dimensiones: (6861, 24)


,Nombre,Edad_Meses,Raza_Principal,Raza_Secundaria,Genero,Color_Principal,Color_Secundario,Color_Terciario,Tamano_Madurez,Longitud_Pelo,...,Cantidad,Cuota_Adopcion,Estado,ID_Rescatista,Cantidad_Videos,Descripcion,ID_Mascota,Cantidad_Fotos,Velocidad_Adopcion,Velocidad_Adopcion_Nombre
0,Nibble,3,299,0,Macho,1,7,0,Pequeño,Corto,...,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1,2,8–30 días
1,No Name Yet,1,265,0,Macho,1,2,0,Mediano,Medio,...,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2,0,Mismo día
2,Sin nombre,3,266,0,Hembra,5,6,0,Mediano,Corto,...,1,0,41326,22fe332bf9c924d4718005891c63fbed,0,This is a stray kitten that came to my house. ...,d24c30b4b,2,2,8–30 días
3,BULAT,12,264,264,Macho,1,0,0,Mediano,Largo,...,1,300,41326,1e0b5a458b5b77f5af581d57ebf570b3,0,anyone within the area of ipoh or taiping who ...,1caa6fcdb,3,1,1–7 días
4,Sin nombre,2,265,0,Hembra,6,0,0,Mediano,Medio,...,1,0,41326,d8af7afece71334473575c9f70daf00d,0,"healthy and active, feisty kitten found in nei...",c06d167ca,6,1,1–7 días


#### Carga de diccionarios

In [3]:
razas = pd.read_csv("breed_labels.csv")

In [4]:
colores = pd.read_csv("color_labels.csv")

In [5]:
estados = pd.read_csv("state_labels.csv")

#### Correcciones al dataset: "breed_labels.csv"

In [6]:
print(razas.shape)
print(razas.columns)
display(razas.head())

(307, 3)
Index(['BreedID', 'Type', 'BreedName'], dtype='object')


,BreedID,Type,BreedName
0,"1,1,""Affenpinscher""",NaN,NaN
1,"2,1,""Afghan Hound""",NaN,NaN
2,"3,1,""Airedale Terrier""",NaN,NaN
3,"4,1,""Akbash""",NaN,NaN
4,"5,1,""Akita""",NaN,NaN


In [7]:
# Separar los datos que están dentro de BreedID
razas[["BreedID", "Type", "BreedName"]] = (razas["BreedID"].str.split(",", expand=True))

# Limpiar BreedName
razas["BreedName"] = (razas["BreedName"].str.replace('"', '', regex=False).str.replace(r"\s+", " ", regex=True).str.strip())

# Convertir a numérico
razas["BreedID"] = pd.to_numeric(razas["BreedID"])
razas["Type"] = pd.to_numeric(razas["Type"])

print(razas.shape)
display(razas.head())

(307, 3)


,BreedID,Type,BreedName
0,1,1,Affenpinscher
1,2,1,Afghan Hound
2,3,1,Airedale Terrier
3,4,1,Akbash
4,5,1,Akita


#### Mejoras a los diccionarios

In [8]:
# Diccionario de razas
razas_gatos = (razas[razas["Type"] == 2].loc[:, ["BreedID", "BreedName"]].rename(columns={"BreedName": "Nombre_Raza"}))

# Diccionario de colores
traduccion_colores = {
    "Black": "Negro",
    "Brown": "Marrón",
    "Golden": "Dorado",
    "Yellow": "Amarillo",
    "Cream": "Crema",
    "Gray": "Gris",
    "White": "Blanco"
}

colores["Nombre_Color"] = colores["ColorName"].replace(traduccion_colores)
colores = colores[["ColorID", "Nombre_Color"]]

# Diccionario de estados
estados = estados.rename(columns={"StateName": "Nombre_Estado"})

# Resumen
print("Razas de gatos:", razas_gatos.shape[0])
print("Colores:", colores.shape[0])
print("Estados:", estados.shape[0])

display(razas_gatos.head())
display(colores)
display(estados.head())

Razas de gatos: 66
Colores: 7
Estados: 15


,BreedID,Nombre_Raza
241,241,Abyssinian
242,242,American Curl
243,243,American Shorthair
244,244,American Wirehair
245,245,Applehead Siamese


,ColorID,Nombre_Color
0,1,Negro
1,2,Marrón
2,3,Dorado
3,4,Amarillo
4,5,Crema
5,6,Gris
6,7,Blanco


,StateID,Nombre_Estado
0,41336,Johor
1,41325,Kedah
2,41367,Kelantan
3,41401,Kuala Lumpur
4,41415,Labuan


#### Integración de diccionarios

In [9]:
# Integración de la RAZA PRINCIPAL
df_dic = df_limpio.copy()

df_dic = df_dic.merge(razas_gatos,how="left",left_on="Raza_Principal",right_on="BreedID")
df_dic.rename(columns={"Nombre_Raza": "Raza_Principal_Nombre"}, inplace=True)
df_dic.drop(columns="BreedID", inplace=True)

display(df_dic[["Raza_Principal","Raza_Principal_Nombre"]].head())

,Raza_Principal,Raza_Principal_Nombre
0,299,Tabby
1,265,Domestic Medium Hair
2,266,Domestic Short Hair
3,264,Domestic Long Hair
4,265,Domestic Medium Hair


Se incorporó una nueva columna denominada **Raza_Principal_Nombre**, que contiene el nombre descriptivo de la raza asociado al código de la raza principal de cada gato.

In [10]:
# Integración de la RAZA SECUNDARIA
df_dic = df_dic.merge(razas_gatos,how="left",left_on="Raza_Secundaria",right_on="BreedID")

df_dic.rename(columns={"Nombre_Raza": "Raza_Secundaria_Nombre"}, inplace=True)
df_dic.drop(columns="BreedID", inplace=True)

# Reemplazar los valores sin segunda raza
df_dic["Raza_Secundaria_Nombre"] = (
    df_dic["Raza_Secundaria_Nombre"]
    .fillna("Sin segunda raza")
)

display(df_dic[["Raza_Secundaria","Raza_Secundaria_Nombre"]].head())

,Raza_Secundaria,Raza_Secundaria_Nombre
0,0,Sin segunda raza
1,0,Sin segunda raza
2,0,Sin segunda raza
3,264,Domestic Long Hair
4,0,Sin segunda raza


In [11]:
# Integración del Color Principal
df_dic = df_dic.merge(colores,how="left",left_on="Color_Principal",right_on="ColorID")
df_dic.rename(columns={"Nombre_Color": "Color_Principal_Nombre"}, inplace=True)
df_dic.drop(columns="ColorID", inplace=True)

In [12]:
# Integración del Color Secundario
df_dic = df_dic.merge(colores,how="left",left_on="Color_Secundario",right_on="ColorID")

df_dic.rename(columns={"Nombre_Color": "Color_Secundario_Nombre"}, inplace=True)
df_dic.drop(columns="ColorID", inplace=True)

df_dic["Color_Secundario_Nombre"] = (df_dic["Color_Secundario_Nombre"].fillna("Sin color secundario"))

In [13]:
# Integración del Color Terciario
df_dic = df_dic.merge(colores,how="left",left_on="Color_Terciario",right_on="ColorID")

df_dic.rename(columns={"Nombre_Color": "Color_Terciario_Nombre"}, inplace=True)
df_dic.drop(columns="ColorID", inplace=True)

df_dic["Color_Terciario_Nombre"] = (df_dic["Color_Terciario_Nombre"].fillna("Sin color terciario"))

In [14]:
display(df_dic[[
    "Color_Principal",
    "Color_Principal_Nombre",
    "Color_Secundario",
    "Color_Secundario_Nombre",
    "Color_Terciario",
    "Color_Terciario_Nombre"
]].head())

,Color_Principal,Color_Principal_Nombre,Color_Secundario,Color_Secundario_Nombre,Color_Terciario,Color_Terciario_Nombre
0,1,Negro,7,Blanco,0,Sin color terciario
1,1,Negro,2,Marrón,0,Sin color terciario
2,5,Crema,6,Gris,0,Sin color terciario
3,1,Negro,0,Sin color secundario,0,Sin color terciario
4,6,Gris,0,Sin color secundario,0,Sin color terciario


In [15]:
# Integración de estados
df_dic = df_dic.merge(estados,how="left",left_on="Estado",right_on="StateID")

df_dic.rename(columns={"Nombre_Estado": "Estado_Nombre"}, inplace=True)
df_dic.drop(columns="StateID", inplace=True)

display(df_dic[["Estado","Estado_Nombre"]].head())

,Estado,Estado_Nombre
0,41326,Selangor
1,41401,Kuala Lumpur
2,41326,Selangor
3,41326,Selangor
4,41326,Selangor


#### Verificación del dataset

In [16]:
print("Dimensiones del dataset:", df_dic.shape)

print("\nNuevas columnas incorporadas:")
columnas_diccionario = [
    "Raza_Principal_Nombre",
    "Raza_Secundaria_Nombre",
    "Color_Principal_Nombre",
    "Color_Secundario_Nombre",
    "Color_Terciario_Nombre",
    "Estado_Nombre"
]

display(df_dic[columnas_diccionario].head())

print("\nValores nulos en las nuevas columnas:")
display(df_dic[columnas_diccionario].isnull().sum())

Dimensiones del dataset: (6861, 30)

Nuevas columnas incorporadas:


,Raza_Principal_Nombre,Raza_Secundaria_Nombre,Color_Principal_Nombre,Color_Secundario_Nombre,Color_Terciario_Nombre,Estado_Nombre
0,Tabby,Sin segunda raza,Negro,Blanco,Sin color terciario,Selangor
1,Domestic Medium Hair,Sin segunda raza,Negro,Marrón,Sin color terciario,Kuala Lumpur
2,Domestic Short Hair,Sin segunda raza,Crema,Gris,Sin color terciario,Selangor
3,Domestic Long Hair,Domestic Long Hair,Negro,Sin color secundario,Sin color terciario,Selangor
4,Domestic Medium Hair,Sin segunda raza,Gris,Sin color secundario,Sin color terciario,Selangor



Valores nulos en las nuevas columnas:


Raza_Principal_Nombre      13
Raza_Secundaria_Nombre      0
Color_Principal_Nombre      0
Color_Secundario_Nombre     0
Color_Terciario_Nombre      0
Estado_Nombre               0
dtype: int64

#### Detección de registros sin correspondencia

Después de integrar el diccionario de razas, se verifica si existen registros cuya raza principal no pudo asociarse con un nombre descriptivo.

In [17]:
# Registros que no encontraron una raza en el diccionario
razas_sin_correspondencia = df_dic[df_dic["Raza_Principal_Nombre"].isna()]
print(f"Cantidad de registros sin correspondencia: {len(razas_sin_correspondencia)}")
display(razas_sin_correspondencia[["ID_Mascota","Nombre","Raza_Principal","Edad_Meses"]])

Cantidad de registros sin correspondencia: 13


,ID_Mascota,Nombre,Raza_Principal,Edad_Meses
10,1bc0f89d8,"Kenit, Kenot, Techit, Keyad, Owen",114,0
712,15a206d0d,Shuka,25,3
730,f8654865f,Mi Cai 2,21,1
1457,27e74e45c,Sin nombre,0,3
1742,36b20cfb5,Sin nombre,25,3
1936,699a81c51,Mo-Joe,218,1
3048,85ec1aac0,Munchi,15,1
3347,6a72cfda7,Mao Mao,70,1
4083,6c399cb06,Bobby The Smiling Shih Tzu,205,36
5006,504134fd6,Kittens Encik Faisal,307,3


Se identificaron **13 registros** cuya `Raza_Principal` no encontró una correspondencia dentro del diccionario de razas de gatos. Antes de modificar estos registros, se realizará un análisis de los códigos involucrados.

#### Análisis de códigos sin correspondencia

Se contabilizan los códigos de raza que no fueron encontrados para identificar si corresponden a categorías especiales o inconsistencias del conjunto de datos.

In [18]:
codigos_problematicos = (razas_sin_correspondencia["Raza_Principal"].value_counts().rename_axis("Codigo_Raza").reset_index(name="Frecuencia"))
display(codigos_problematicos)

,Codigo_Raza,Frecuencia
0,307,4
1,25,2
2,114,1
3,21,1
4,0,1
5,218,1
6,15,1
7,70,1
8,205,1


#### Actualización del diccionario de razas

In [19]:
# Agregación de categorías especiales al diccionario
excepciones_razas = pd.DataFrame({
    "BreedID": [0, 307, 15, 21, 25, 70, 114, 205, 218],
    "Nombre_Raza": [
        "Raza no especificada",
        "Raza mixta",
        "Raza de perro registrada",
        "Raza de perro registrada",
        "Raza de perro registrada",
        "Raza de perro registrada",
        "Raza de perro registrada",
        "Raza de perro registrada",
        "Raza de perro registrada"
    ]
})

# Añadir excepciones al diccionario de gatos
razas_gatos = pd.concat([razas_gatos, excepciones_razas],ignore_index=True)

# Evitar códigos repetidos
razas_gatos = razas_gatos.drop_duplicates(subset="BreedID")
print("Total de categorías de razas:", len(razas_gatos))
display(razas_gatos.tail(10))

Total de categorías de razas: 75


,BreedID,Nombre_Raza
65,306,Tuxedo
66,0,Raza no especificada
67,307,Raza mixta
68,15,Raza de perro registrada
69,21,Raza de perro registrada
70,25,Raza de perro registrada
71,70,Raza de perro registrada
72,114,Raza de perro registrada
73,205,Raza de perro registrada
74,218,Raza de perro registrada


#### Reintegración del diccionario

In [20]:
# Eliminar la columna anterior para volver a crearla
df_dic = df_dic.drop(columns="Raza_Principal_Nombre")
# Integración nuevamente
df_dic = df_dic.merge(razas_gatos,how="left",left_on="Raza_Principal",right_on="BreedID")
df_dic.rename(columns={"Nombre_Raza":"Raza_Principal_Nombre"}, inplace=True)
df_dic.drop(columns="BreedID", inplace=True)

#### Validación final

In [21]:
print("="*60)
print("VALIDACIÓN FINAL DE DICCIONARIOS")
print("="*60)

columnas_diccionario = [
    "Raza_Principal_Nombre",
    "Raza_Secundaria_Nombre",
    "Color_Principal_Nombre",
    "Color_Secundario_Nombre",
    "Color_Terciario_Nombre",
    "Estado_Nombre"
]

validacion = pd.DataFrame({
    "Columna": columnas_diccionario,
    "Valores nulos": [
        df_dic[col].isna().sum()
        for col in columnas_diccionario
    ]
})

display(validacion)

VALIDACIÓN FINAL DE DICCIONARIOS


,Columna,Valores nulos
0,Raza_Principal_Nombre,0
1,Raza_Secundaria_Nombre,0
2,Color_Principal_Nombre,0
3,Color_Secundario_Nombre,0
4,Color_Terciario_Nombre,0
5,Estado_Nombre,0


### Conclusiones de la incorporación de diccionarios

En esta etapa se integraron correctamente los diccionarios de razas, colores y estados al conjunto de datos de gatos de PetFinder (dataset.gatos.limpio.csv) mediante operaciones `merge` de Pandas.

Como resultado:

* Se incorporaron nombres descriptivos para las razas principales y secundarias.
* Se reemplazaron los códigos de colores por sus nombres en español.
* Se añadieron los nombres de los estados de Malasia asociados a cada registro.
* Se mantuvieron los códigos originales junto con las nuevas columnas descriptivas, permitiendo conservar la trazabilidad del conjunto de datos.

El dataset pasó de **23** a **30 variables**, quedando preparado para el análisis exploratorio y la construcción del dashboard en Power BI.


#### Exportación del dataset limpio y enriquecido

In [22]:
df_dic.to_csv("dataset_gatos_preparado.csv",index=False,encoding="utf-8-sig")

print("✅ Dataset exportado correctamente.")
print("Archivo: dataset_gatos_preparado.csv")
print(f"Dimensiones: {df_dic.shape}")

✅ Dataset exportado correctamente.
Archivo: dataset_gatos_preparado.csv
Dimensiones: (6861, 30)


In [23]:
df_dic.describe()

,Edad_Meses,Raza_Principal,Raza_Secundaria,Color_Principal,Color_Secundario,Color_Terciario,Cantidad,Cuota_Adopcion,Estado,Cantidad_Videos,Cantidad_Fotos,Velocidad_Adopcion
count,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000,6861.000000
mean,7.543361,269.046640,68.175193,2.419764,3.958315,2.537239,1.634456,17.663169,41350.427489,0.054511,4.076665,2.399504
std,12.771554,14.209506,117.924234,1.892870,2.680185,3.232753,1.390996,61.603234,34.453207,0.323432,3.490296,1.205065
min,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,41324.000000,0.000000,0.000000,0.000000
25%,2.000000,265.000000,0.000000,1.000000,2.000000,0.000000,1.000000,0.000000,41326.000000,0.000000,2.000000,1.000000
50%,3.000000,266.000000,0.000000,1.000000,4.000000,0.000000,1.000000,0.000000,41326.000000,0.000000,3.000000,2.000000
75%,8.000000,266.000000,241.000000,3.000000,7.000000,7.000000,2.000000,0.000000,41401.000000,0.000000,5.000000,4.000000
max,212.000000,307.000000,306.000000,7.000000,7.000000,7.000000,20.000000,800.000000,41415.000000,6.000000,30.000000,4.000000
